# Gaussian Mixture Models — soft, elliptical clustering via EM

> Tutorial pair for [`gaussian_mixture.py`](gaussian_mixture.py).

## 1. Intuition
k-means draws hard, spherical boundaries. A **Gaussian Mixture Model** says the
data was generated by drawing, for each point, a hidden cluster label and then
sampling from that cluster's Gaussian. We don't see the labels, so we infer
*soft* memberships (responsibilities) and fit each cluster's mean **and**
covariance — clusters can be elongated, tilted ellipses of different sizes.

## 2. Concept (the slide)
- **Model:** $p(\mathbf x)=\sum_{k=1}^K \pi_k\,\mathcal N(\mathbf x\mid\boldsymbol\mu_k,\Sigma_k)$
  with mixing weights $\pi_k\ge 0,\ \sum_k\pi_k=1$.
- **Latent variable:** a one-hot $z$ saying which component generated $\mathbf x$.
- **Fit by EM:** the log-likelihood is not concave (sum inside the log), but EM
  alternates a tractable **E-step** (posterior over $z$) and **M-step**
  (weighted Gaussian MLE), each guaranteed not to decrease the likelihood.
- **Covariance type:** *full* (any ellipse) vs *diagonal* (axis-aligned, fewer
  params). k-means $\approx$ GMM with shared spherical $\Sigma=\sigma^2 I$ and
  hardened responsibilities.

## 3. Math derivation

**Incomplete-data log-likelihood** for data $X=\{\mathbf x_i\}$:
$$\ell(\theta)=\sum_{i=1}^n\log\sum_{k=1}^K \pi_k\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k).$$
The $\log\sum$ couples the parameters; direct maximization is hard.

**The ELBO (Jensen lower bound).** Introduce any distribution $q_i(k)$ over the
latent label of point $i$. By Jensen's inequality (log is concave),
$$\ell(\theta)=\sum_i\log\sum_k q_i(k)\frac{\pi_k\mathcal N(\mathbf x_i\mid\theta_k)}{q_i(k)}
\ \ge\ \sum_i\sum_k q_i(k)\log\frac{\pi_k\mathcal N(\mathbf x_i\mid\theta_k)}{q_i(k)}
\ \equiv\ \mathcal L(q,\theta).$$
The gap is exactly $\ell(\theta)-\mathcal L(q,\theta)=\sum_i \mathrm{KL}\!\big(q_i\,\|\,p(z_i\mid\mathbf x_i,\theta)\big)\ge 0$.

**E-step** maximizes $\mathcal L$ over $q$ with $\theta$ fixed: the KL is zero
when $q_i(k)=p(z_i=k\mid\mathbf x_i,\theta)$, the **responsibility**
$$\gamma_{ik}=\frac{\pi_k\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k)}
{\sum_j \pi_j\,\mathcal N(\mathbf x_i\mid\boldsymbol\mu_j,\Sigma_j)}.$$
Now the bound *touches* the likelihood: $\mathcal L(q,\theta)=\ell(\theta)$.

**M-step** maximizes $\mathcal L$ over $\theta$ with $q=\gamma$ fixed. Drop the
$-q\log q$ entropy term (constant in $\theta$) and maximize
$Q(\theta)=\sum_i\sum_k \gamma_{ik}\big[\log\pi_k+\log\mathcal N(\mathbf x_i\mid\boldsymbol\mu_k,\Sigma_k)\big]$.
Setting gradients to zero (with a Lagrange multiplier for $\sum_k\pi_k=1$) gives
the **weighted MLE**, with $N_k=\sum_i\gamma_{ik}$:
$$\boldsymbol\mu_k=\frac1{N_k}\sum_i\gamma_{ik}\mathbf x_i,\quad
\Sigma_k=\frac1{N_k}\sum_i\gamma_{ik}(\mathbf x_i-\boldsymbol\mu_k)(\mathbf x_i-\boldsymbol\mu_k)^\top,\quad
\pi_k=\frac{N_k}{n}.$$

**Why EM monotonically increases $\ell$.** After the E-step,
$\ell(\theta^t)=\mathcal L(q^{t+1},\theta^t)$. The M-step picks $\theta^{t+1}$ to
*maximize* $\mathcal L(q^{t+1},\cdot)$, so $\mathcal L(q^{t+1},\theta^{t+1})\ge\mathcal L(q^{t+1},\theta^t)$.
And $\ell(\theta^{t+1})\ge\mathcal L(q^{t+1},\theta^{t+1})$ because the bound is
a lower bound. Chaining:
$$\ell(\theta^{t+1})\ \ge\ \mathcal L(q^{t+1},\theta^{t+1})\ \ge\ \mathcal L(q^{t+1},\theta^t)\ =\ \ell(\theta^t).$$
So the likelihood never decreases; bounded above, the sequence converges (to a
stationary point — possibly a local max, hence multiple restarts).

**Practical notes.** Compute $\log\mathcal N$ with a Cholesky factor
$\Sigma=LL^\top$ (so $\log\det\Sigma=2\sum_i\log L_{ii}$ and the Mahalanobis term
is $\lVert L^{-1}(\mathbf x-\boldsymbol\mu)\rVert^2$), normalize responsibilities
with log-sum-exp, and add $\varepsilon I$ to $\Sigma_k$ to avoid singular
("collapsing") components.

## 4. NumPy implementation (full + diagonal covariance, BIC selection)

In [ ]:
# ===== actual implementation from gaussian_mixture.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _logsumexp(a, axis=None, keepdims=False):
    """Numerically stable log(sum(exp(a)))."""
    amax = np.max(a, axis=axis, keepdims=True)
    out = np.log(np.sum(np.exp(a - amax), axis=axis, keepdims=True))
    out = out + amax
    if not keepdims and axis is not None:
        out = np.squeeze(out, axis=axis)
    return out

import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import make_blobs
    from sklearn.metrics import adjusted_rand_score

    # anisotropic blobs: GMM (elliptical) should beat spherical k-means here.
    X, ytrue = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=SEED)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    X = X @ transform

    for cov in ("full", "diag"):
        gmm = GMMNumPy(n_components=3, covariance_type=cov).fit(X)
        ari = adjusted_rand_score(ytrue, gmm.labels_)
        mono = all(b - a >= -1e-6 for a, b in zip(gmm.history_, gmm.history_[1:]))
        print(f"NumPy GMM ({cov:4s}): ARI={ari:.3f}  logL/n={gmm.score(X):.3f}  "
              f"BIC={gmm.bic(X):.1f}  monotone_LL={mono}")

    gt = GMMTorch(n_components=3).fit(X)
    print(f"Torch GMM (full): ARI={adjusted_rand_score(ytrue, gt.labels_):.3f}  "
          f"logL={gt.log_likelihood_:.1f}  device={gt.device}")

    print("\nModel selection by BIC (full covariance):")
    for k in range(2, 6):
        g = GMMNumPy(n_components=k, covariance_type="full").fit(X)
        print(f"  K={k}: BIC={g.bic(X):.1f}")


class GMMNumPy:
    r"""
    A mixture of K Gaussians:

        p(x) = sum_k  pi_k  N(x | mu_k, Sigma_k),     sum_k pi_k = 1.

    EM introduces latent assignments z_i in {1..K} and alternates:

      E-step: responsibilities  gamma_ik = p(z_i=k | x_i)
                = pi_k N(x_i|mu_k,Sig_k) / sum_j pi_j N(x_i|mu_j,Sig_j).

      M-step: weighted MLE with weights gamma_ik:
                N_k  = sum_i gamma_ik
                mu_k = (1/N_k) sum_i gamma_ik x_i
                Sig_k= (1/N_k) sum_i gamma_ik (x_i-mu_k)(x_i-mu_k)^T
                pi_k = N_k / n.

    Each full E+M step never decreases the data log-likelihood.
    """

    def __init__(self, n_components=3, covariance_type="full", n_iters=200,
                 tol=1e-6, reg_covar=1e-6, n_init=3, seed=SEED):
        self.K = n_components
        self.covariance_type = covariance_type
        self.n_iters, self.tol = n_iters, tol
        self.reg_covar, self.n_init, self.seed = reg_covar, n_init, seed

    # --- log N(x | mu, Sigma) for every point/component, shape (n, K) ----
    def _log_gauss(self, X, means, covs):
        n, d = X.shape
        out = np.empty((n, self.K))
        log2pi = d * np.log(2 * np.pi)
        for k in range(self.K):
            diff = X - means[k]                       # (n, d)
            if self.covariance_type == "diag":
                var = covs[k]                         # (d,) diagonal variances
                # (x-mu)^T Sig^{-1} (x-mu) = sum_j diff_j^2 / var_j
                maha = (diff * diff / var).sum(1)
                logdet = np.log(var).sum()
            else:                                     # full covariance
                # solve via Cholesky for stability:  Sig = L L^T
                L = np.linalg.cholesky(covs[k])
                # solve L y = diff^T  -> maha = ||y||^2 , logdet = 2 sum log L_ii
                y = np.linalg.solve(L, diff.T)        # (d, n)
                maha = (y * y).sum(0)
                logdet = 2.0 * np.log(np.diag(L)).sum()
            out[:, k] = -0.5 * (log2pi + logdet + maha)
        return out

    def _init_means_kpp(self, X, rng):
        # k-means++ seeding: spread the initial means out by D^2 sampling.
        means = [X[rng.integers(len(X))]]
        for _ in range(1, self.K):
            d2 = np.min([((X - m) ** 2).sum(1) for m in means], axis=0)
            probs = d2 / d2.sum()
            means.append(X[rng.choice(len(X), p=probs)])
        return np.array(means, float)

    def _init_params(self, X, rng):
        n, d = X.shape
        means = self._init_means_kpp(X, rng)
        weights = np.full(self.K, 1.0 / self.K)
        gvar = X.var(0) + self.reg_covar               # global spread
        if self.covariance_type == "diag":
            covs = np.tile(gvar, (self.K, 1))
        else:
            covs = np.array([np.diag(gvar) for _ in range(self.K)])
        return weights, means, covs

    def _e_step(self, X, weights, means, covs):
        # log-responsibilities, normalized with the log-sum-exp trick.
        log_w = np.log(weights + 1e-300)
        log_p = self._log_gauss(X, means, covs) + log_w   # (n, K) joint log p(x,z)
        log_norm = _logsumexp(log_p, axis=1, keepdims=True)  # log p(x)
        log_gamma = log_p - log_norm
        ll = log_norm.sum()                               # incomplete-data LL
        return np.exp(log_gamma), ll

    def _m_step(self, X, gamma):
        n, d = X.shape
        Nk = gamma.sum(0) + 1e-300                         # (K,) soft counts
        weights = Nk / n
        means = (gamma.T @ X) / Nk[:, None]                # (K, d)
        if self.covariance_type == "diag":
            covs = np.empty((self.K, d))
            for k in range(self.K):
                diff = X - means[k]
                covs[k] = (gamma[:, k, None] * diff * diff).sum(0) / Nk[k] + self.reg_covar
        else:
            covs = np.empty((self.K, d, d))
            for k in range(self.K):
                diff = X - means[k]                        # (n, d)
                covs[k] = (gamma[:, k, None] * diff).T @ diff / Nk[k]
                covs[k] += self.reg_covar * np.eye(d)      # regularize
        return weights, means, covs

    def _fit_once(self, X, rng):
        weights, means, covs = self._init_params(X, rng)
        prev_ll = -np.inf
        history = []
        for _ in range(self.n_iters):
            gamma, ll = self._e_step(X, weights, means, covs)
            history.append(ll)
            weights, means, covs = self._m_step(X, gamma)
            if ll - prev_ll < self.tol and prev_ll > -np.inf:
                break
            prev_ll = ll
        # final responsibilities/LL after last M-step
        gamma, ll = self._e_step(X, weights, means, covs)
        return weights, means, covs, gamma, ll, history

    def fit(self, X):
        X = np.asarray(X, float)
        best_ll = -np.inf
        for i in range(self.n_init):
            rng = np.random.default_rng(self.seed + i)
            w, m, c, g, ll, hist = self._fit_once(X, rng)
            if ll > best_ll:
                best_ll = ll
                self.weights_, self.means_, self.covariances_ = w, m, c
                self.responsibilities_ = g
                self.log_likelihood_ = ll
                self.history_ = hist
        self.labels_ = self.responsibilities_.argmax(1)
        return self

    def predict_proba(self, X):
        g, _ = self._e_step(np.asarray(X, float), self.weights_,
                            self.means_, self.covariances_)
        return g

    def predict(self, X):
        return self.predict_proba(X).argmax(1)

    def score(self, X):
        _, ll = self._e_step(np.asarray(X, float), self.weights_,
                            self.means_, self.covariances_)
        return ll / len(X)

    def bic(self, X):
        # BIC = -2 LL + p log n ; rewards fit, penalizes #free parameters p.
        X = np.asarray(X, float)
        n, d = X.shape
        if self.covariance_type == "diag":
            cov_params = self.K * d
        else:
            cov_params = self.K * d * (d + 1) // 2
        p = (self.K - 1) + self.K * d + cov_params          # weights+means+cov
        ll = self.log_likelihood_
        return -2 * ll + p * np.log(n)

## 5. PyTorch implementation (vectorized closed-form EM, GPU-friendly)

In [ ]:
# ===== actual implementation from gaussian_mixture.py =====
class GMMTorch:
    """Full-covariance EM on torch tensors (no autograd; closed-form updates)."""

    def __init__(self, n_components=3, n_iters=200, tol=1e-6, reg_covar=1e-6, seed=SEED):
        self.K, self.n_iters = n_components, n_iters
        self.tol, self.reg_covar, self.seed = tol, reg_covar, seed
        self.device = get_device()

    def _log_gauss(self, X, means, covs):
        n, d = X.shape
        log2pi = d * np.log(2 * np.pi)
        out = torch.empty((n, self.K), device=X.device, dtype=X.dtype)
        for k in range(self.K):
            diff = X - means[k]
            L = torch.linalg.cholesky(covs[k])
            y = torch.linalg.solve_triangular(L, diff.T, upper=False)  # (d,n)
            maha = (y * y).sum(0)
            logdet = 2.0 * torch.log(torch.diagonal(L)).sum()
            out[:, k] = -0.5 * (log2pi + logdet + maha)
        return out

    def fit(self, X):
        torch.manual_seed(self.seed)
        Xt = torch.as_tensor(np.asarray(X, float), dtype=torch.float64, device=self.device)
        n, d = Xt.shape
        # init means at random distinct points; covs = global covariance.
        idx = torch.randperm(n, device=self.device)[:self.K]
        means = Xt[idx].clone()
        gvar = Xt.var(0, unbiased=False)
        covs = torch.stack([torch.diag(gvar) for _ in range(self.K)])
        weights = torch.full((self.K,), 1.0 / self.K, dtype=Xt.dtype, device=self.device)
        prev = -float("inf")
        for _ in range(self.n_iters):
            # E-step
            log_p = self._log_gauss(Xt, means, covs) + torch.log(weights + 1e-300)
            log_norm = torch.logsumexp(log_p, dim=1, keepdim=True)
            gamma = torch.exp(log_p - log_norm)
            ll = log_norm.sum().item()
            # M-step
            Nk = gamma.sum(0) + 1e-300
            weights = Nk / n
            means = (gamma.T @ Xt) / Nk[:, None]
            covs = torch.empty((self.K, d, d), dtype=Xt.dtype, device=self.device)
            for k in range(self.K):
                diff = Xt - means[k]
                covs[k] = (gamma[:, k, None] * diff).T @ diff / Nk[k]
                covs[k] += self.reg_covar * torch.eye(d, dtype=Xt.dtype, device=self.device)
            if abs(ll - prev) < self.tol and prev > -float("inf"):
                break
            prev = ll
        self.weights_ = weights.cpu().numpy()
        self.means_ = means.cpu().numpy()
        self.covariances_ = covs.cpu().numpy()
        self.labels_ = gamma.argmax(1).cpu().numpy()
        self.log_likelihood_ = ll
        return self

## 6. Train / run — ARI, monotone log-likelihood, BIC over K

In [ ]:
demo()

## 7. Visualization — soft clusters, covariance ellipses, EM convergence

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.datasets import make_blobs
import gaussian_mixture as M

X, y = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=0)
X = X @ np.array([[0.6, -0.6], [-0.4, 0.8]])   # make blobs anisotropic
gmm = M.GMMNumPy(n_components=3, covariance_type="full").fit(X)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:, 0], X[:, 1], c=gmm.labels_, s=10, cmap="tab10")
ax[0].scatter(gmm.means_[:, 0], gmm.means_[:, 1], c="k", marker="X", s=160)
for k in range(gmm.K):                          # draw 2-sigma covariance ellipse
    vals, vecs = np.linalg.eigh(gmm.covariances_[k])
    ang = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    w, h = 2 * 2 * np.sqrt(vals[::-1])
    ax[0].add_patch(Ellipse(gmm.means_[k], w, h, angle=ang, fill=False, edgecolor="k", lw=2))
ax[0].set_title("GMM full covariance (2-sigma ellipses)")

ax[1].plot(gmm.history_, "o-")
ax[1].set_xlabel("EM iteration"); ax[1].set_ylabel("log-likelihood")
ax[1].set_title("EM monotonically increases logL")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- EM only finds a **local** optimum and is init-sensitive → use k-means++ seeding
  and **multiple restarts** (keep the best log-likelihood).
- Without `reg_covar`, a component can collapse onto a single point
  ($\Sigma\to 0$, likelihood $\to\infty$): a singularity, not a real solution.
- **Full** covariance is flexible but $O(d^2)$ params per component; **diagonal**
  is cheaper and robust in high dimensions but only axis-aligned ellipses.
- Choose $K$ with **BIC/AIC** (penalized likelihood), not raw log-likelihood
  (which always improves with more components).

**Next:** drop the probabilistic model and cluster by *density* → DBSCAN.